# QLoRA Fine-Tuning on Google Colab - Notes

## 1. Set up Hugging Face PEFT + Transformers + bitsandbytes

### What is Google Colab?

Google Colab is an online Python notebook that can provide access to a GPU.

We use Colab because LLM fine-tuning needs a lot of computing power and GPU memory.

```text
Google Colab
    ↓
GPU
    ↓
LLM
    ↓
QLoRA Fine-Tuning
```

### What is Transformers?

Transformers is a Python library from Hugging Face.

It helps us:

* Load pretrained LLMs
* Load tokenizers
* Generate text
* Fine-tune models

Example:

```python
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM
```

Remember:

```text
Transformer = AI architecture

Transformers = Python library
```

### What is PEFT?

PEFT = Parameter-Efficient Fine-Tuning.

It allows us to fine-tune a model without updating all of its parameters.

```text
Full Fine-Tuning
    ↓
Update almost all model parameters
    ↓
High GPU memory
```

PEFT:

```text
Base Model
    ↓
Keep most parameters frozen
    ↓
Train small additional parameters
    ↓
Less memory
```

LoRA is one type of PEFT method.

```text
PEFT
 ├── LoRA
 ├── Prefix Tuning
 ├── Prompt Tuning
 └── Other methods
```

### What is bitsandbytes?

bitsandbytes is a library commonly used for memory-efficient low-bit model operations.

It allows us to load models using quantization such as 4-bit.

```text
Normal model
    ↓
More memory

4-bit model
    ↓
Less memory
```

### Install libraries

```python
!pip install -U transformers peft bitsandbytes accelerate datasets
```

Check GPU:

```python
import torch

print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
```

---

# 2. Load a Small Base Model

A base model is an already pretrained LLM.

Examples:

```text
TinyLlama
Phi-3-mini
```

The base model already knows general language patterns.

We don't train an LLM from zero.

```text
Pretrained Model
      ↓
QLoRA Fine-Tuning
      ↓
Specialized Model
```

### Example

Suppose we want to create a Python tutor.

```text
TinyLlama
    ↓
Python instruction dataset
    ↓
QLoRA
    ↓
Python Tutor
```

### Select TinyLlama

```python
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
```

### Load tokenizer

```python
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
```

Tokenizer converts text into tokens/token IDs.

```text
"What is Python?"
       ↓
   Tokenizer
       ↓
 [Token IDs]
       ↓
     Model
```

### 4-bit quantization

```python
import torch
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)
```

Meaning:

```text
load_in_4bit=True
→ Load model weights using 4-bit quantization

compute_dtype=torch.float16
→ Use FP16 for computations

nf4
→ 4-bit quantization format commonly used for QLoRA

double_quant=True
→ Additional memory saving
```

### Load model

```python
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)
```

Now:

```text
TinyLlama
    ↓
Transformers
    ↓
bitsandbytes
    ↓
4-bit quantization
    ↓
GPU
```

---

# 3. Prepare Custom Instruction Dataset

The dataset contains examples that teach the model your specific task.

For example, Python Tutor:

```text
Question:
What is a Python variable?

Answer:
A variable is a name used to store a value in Python.
```

We can use **Alpaca format**.

Alpaca format commonly contains:

```text
instruction
input
output
```

### Meaning

```text
instruction
→ What should the model do?

input
→ Additional information/question

output
→ Correct answer
```

### Example

```json
{
  "instruction": "Explain Python variables.",
  "input": "Explain it to a beginner.",
  "output": "A variable is a name used to store a value in Python."
}
```

### Dataset flow

```text
Instruction
     +
Input
     ↓
Expected Output
     ↓
Training Example
```

For this learning project:

```text
50–100 good examples
```

are enough to practice the complete workflow.

Important:

50–100 examples are enough for learning the process, but not necessarily enough for a production-quality model.

---

# 4. Load Dataset in Python

Example:

```python
data = [
    {
        "instruction": "What is Python?",
        "input": "Explain it for a beginner.",
        "output": "Python is a programming language used to build applications and automate tasks."
    },
    {
        "instruction": "What is a Python variable?",
        "input": "Give a simple explanation.",
        "output": "A variable is a name used to store a value in Python."
    }
]
```

Convert it into a Hugging Face Dataset:

```python
from datasets import Dataset

dataset = Dataset.from_list(data)

print(dataset)
```

Output will look like:

```text
Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 2
})
```

For the real experiment, expand the dataset to around 50–100 quality examples.

---

# 5. What is QLoRA?

QLoRA means:

```text
Q = Quantization
LoRA = Low-Rank Adaptation

QLoRA = Quantization + LoRA
```

The basic idea:

```text
Large Base Model
      ↓
4-bit Quantization
      ↓
Freeze Base Model
      ↓
Add LoRA Adapter
      ↓
Train LoRA
```

The base model is mostly kept frozen.

Only the small LoRA parameters are trained.

---

# 6. Add LoRA

LoRA is a PEFT method.

Instead of changing the original model weights directly:

```text
Original weights
      ↓
Keep frozen
```

LoRA adds small trainable matrices.

```text
Original Model
      +
LoRA Adapter
      ↓
Fine-Tuned Model
```

The important formula is:

```text
W_new = W + BA
```

Where:

```text
W = original model weight
A = small trainable matrix
B = small trainable matrix
BA = LoRA update
```

So instead of training the huge `W`, we train the much smaller `A` and `B`.

---

# 7. QLoRA Training

Training flow:

```text
Training Dataset
       ↓
Tokenizer
       ↓
Token IDs
       ↓
4-bit Base Model
       ↓
LoRA Adapter
       ↓
Prediction
       ↓
Calculate Loss
       ↓
Calculate Gradients
       ↓
Update LoRA Parameters
       ↓
Repeat
```

The important point:

```text
Base Model → Frozen

LoRA Parameters → Trainable
```

---

# 8. What is Loss?

Loss tells us how different the model's prediction is from the expected answer.

Simple idea:

```text
Good prediction
     ↓
Lower loss

Bad prediction
     ↓
Higher loss
```

Example:

```text
Step       Loss

10         2.8
20         2.5
30         2.2
40         1.9
50         1.7
```

Generally, decreasing training loss means the model is fitting the training data better.

---

# 9. Loss Curve

We can plot training loss:

```text
Loss
 |
3|\
 | \
2|  \
 |   \
1|    \____
 |
 +----------------
       Steps
```

A decreasing curve is generally a good sign.

But:

```text
Low training loss
      ≠
Automatically good model
```

The model can overfit the training data.

Therefore, we should also test it on examples it did not see during training.

---

# 10. Complete QLoRA Workflow

```text
Google Colab
     ↓
Enable GPU
     ↓
Install Transformers + PEFT + bitsandbytes
     ↓
Load TinyLlama / Phi
     ↓
Load Tokenizer
     ↓
4-bit Quantization
     ↓
Prepare Alpaca Dataset
     ↓
Tokenize Dataset
     ↓
Configure LoRA
     ↓
Add LoRA Adapter
     ↓
Run QLoRA Training
     ↓
Calculate Loss
     ↓
Update LoRA Parameters
     ↓
Repeat Training
     ↓
Monitor Loss Curve
     ↓
Evaluate Model
     ↓
Save LoRA Adapter
```

# 11. Easy Memory Trick

```text
Transformers
= Work with the LLM

PEFT
= Efficient fine-tuning

LoRA
= A PEFT method

bitsandbytes
= Low-bit/quantization tool

Quantization
= Use fewer bits → reduce memory

QLoRA
= 4-bit Quantization + LoRA

Alpaca Dataset
= Instruction + Input + Output

Loss
= Measures training error
```

# 12. Real-World Example

Imagine a company wants a **customer-support AI**.

```text
General LLM
     ↓
Already knows English and general knowledge
```

Company creates examples:

```text
Customer:
How do I reset my password?

Expected:
Go to Settings → Security → Reset Password.
```

Create 50–100 similar examples.

Then:

```text
General LLM
     +
Company examples
     ↓
QLoRA
     ↓
Small LoRA Adapter
     ↓
Company Customer-Support AI
```

The base model doesn't need to be completely retrained.

**This is why QLoRA is useful: it allows you to adapt a relatively large model using much less GPU memory than full fine-tuning.**
